# O ano da Copa em cada lance

## Task 2: trazer o ano da Copa para cada lance

## O problema

O feed não tem o ano da Copa; as partidas têm. Join pelo `match_id`.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-broadcast")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"
EVENTOS = "../../data/events"

spark

A célula abaixo já vem pronta: o código do vídeo 4.1 (lances e partidas).

In [ ]:
# vídeo 4.1: os lances e as partidas
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

eventos = spark.read.parquet(EVENTOS).withColumnRenamed("MatchID", "match_id")

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)

## Trazendo o ano

## Juntar tabelas pode custar caro

Para juntar pela chave, as linhas de mesmo `match_id` dos dois lados precisam estar na mesma partição. O jeito geral é
redistribuir os dois lados pela chave: o **shuffle**, que aparece no plano como **`Exchange`** (vídeo 4.3).

Quando um lado é **pequeno**, o Spark pode mandar uma **cópia inteira** dele para cada executor, e o lado grande não
sai do lugar: o **broadcast join**.

Procure no plano:

- o tipo de join;
- se o caminho dos lances (`FileScan parquet`) tem algum `Exchange` até o join;
- o **`BroadcastExchange`** do lado das partidas.

## Por que o Spark escolheu o broadcast?

O Spark faz broadcast sozinho quando estima que um lado tem até `spark.sql.autoBroadcastJoinThreshold` (padrão: 10 MB).

Com os dois lados grandes, o Spark usa o **sort merge join**, com `Exchange` dos dois lados.

## Nenhum lance se perdeu?

O join padrão é o `inner`: um lance sem partida correspondente sumiria sem aviso.

## Para consultar depois

- **Pedindo o broadcast:** `eventos.join(F.broadcast(partidas), ...)` marca um lado para broadcast, mesmo acima do
  limite. Use só quando tiver certeza de que o lado cabe na memória de cada executor.
- **Desligando o automático:** `spark.sql.autoBroadcastJoinThreshold = -1`.
- **Em SQL:** o mesmo pedido é a dica `SELECT /*+ BROADCAST(p) */ ...`.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()